# Pydantic AI with Ollama and MCP Server

This notebook demonstrates how to use Pydantic AI with Ollama models and MCP servers for DNA sequence analysis.

# Install dependencies
```bash
uv pip install pydantic-ai
uv pip install nest-asyncio
```

# Ollama Model is available
```bash
ollama models list
```
Please make sure qwen3.5:4b is available. if not, please run `ollama pull qwen3.5:4b`. Or choose another model and modify the model name in the code.

In [1]:
import nest_asyncio
import asyncio
import time
nest_asyncio.apply()

In [2]:
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider
from pydantic_ai.mcp import FastMCPClient, MCPToolset





ollama_model = OpenAIChatModel(
    model_name='qwen3.5:4b',
    provider=OllamaProvider(base_url='http://localhost:11434/v1'),
)


In [3]:
# Create MCP server connection
server = MCPToolset(FastMCPClient('http://localhost:8000/mcp'))

In [4]:
# Create agent with MCP server tools and proper system prompt
agent_ollama = Agent(
    ollama_model, 
    toolsets=[server],
    system_prompt='''You are a DNA analysis assistant with access to specialized DNA analysis tools via MCP server.

When analyzing a DNA sequence, you should:
1. First call _list_loaded_models to see what models are available
2. Then call dna_multi_model_predict with the DNA sequence and appropriate model names
3. Interpret and explain the results in a comprehensive way, not just only list the results

Available tools should include:
- _list_loaded_models: Lists available DNA analysis models
- dna_multi_model_predict: Predicts DNA sequence properties using multiple models

Always use the tools to provide accurate analysis. Based on the returned results, make reasonable inferences with comprehensive biological functions of this sequence.'''
)

In [5]:
# Analyze DNA sequence using MCP server with proper async context
async def analyze_dna_sequence():
    async with agent_ollama:  # This ensures proper MCP server connection
        result = await agent_ollama.run(
            'What is the function of following DNA sequence? Please analyze it thoroughly using all available models: AGAAAAAACATGACAAGAAATCGATAATAATACAAAAGCTATGATGGTGTGCAATGTCCGTGTGCATGCGTGCACGCATTGCAACCGGCCCAAATCAAGGCCCATCGATCAGTGAATACTCATGGGCCGGCGGCCCACCACCGCTTCATCTCCTCCTCCGACGACGGGAGCACCCCCGCCGCATCGCCACCGACGAGGAGGAGGCCATTGCCGGCGGCGCCCCCGGTGAGCCGCTGCACCACGTCCCTGA'
        )
        return result

# Run the analysis
result = await analyze_dna_sequence()

time.sleep(3)

In [6]:
# Display the comprehensive analysis result
print("=== DNA Sequence Analysis Result ===")
print(result.output)
print("\n=== Usage Statistics ===")
print(result.usage)


=== DNA Sequence Analysis Result ===
## Comprehensive DNA Sequence Analysis

### Predicted Sequence (202 bp, plant):
```
AGAAAAAACATGACAAGAAATCGATAATAATACAAAAGCTATGATGGTGTGCAATGTCCGTGTGCATGCGTGCACGCATTGCAACCGGCCCAAATCAAGGCCCATCGATCAGTGAATACTCATGGGCCGGCGGCCCACCACCGCTTCATCTCCTCCTCCGACGACGGGAGCACCCCCGCCGCATCGCCACCGACGAGGAGGAGGCCATTGCCGGCGGCGCCCCCGGTGAGCCGCTGCACCACGTCCCTGA
```

---

## 📊 Model Prediction Results

| Model | Architecture | Prediction | Confidence |
|-------|-----------|------------|------------|
| **Promoter Model** | DNABERT (plant) | ❌ **Not a Core Promoter** | **93.7%** |
| **Conservation Model** | DNABERT (plant) | ❌ **Not Conserved** | **98.7%** |
| **Open Chromatin Model** | DNAMamba (plant) | ✅ **Fully Open Chromatin** | **94.6%** |

---

## 🔬 Detailed Biological Interpretation

### 1. Chromatin State: FULLY OPEN (94.6% confidence)
This is the most striking finding. The sequence resides in a **fully accessible chromatin environment**, meaning this DNA region is decond

In [7]:
# Alternative approach: Test individual tool calls to debug
async def test_individual_tools():
    async with agent_ollama:
        # First, let's try to list available tools
        print("=== Testing tool availability ===")
        
        # Try to get the agent to use the list models tool
        list_result = await agent_ollama.run(
            'Please list all the available DNA analysis models using the _list_loaded_models tool.'
        )
        print("List models result:")
        print(list_result.output)
        
        return list_result

# Run individual tool test
list_result = await test_individual_tools()

=== Testing tool availability ===


List models result:
## Available DNA Analysis Models

There are **3 models** currently loaded and ready for use:

---

### 1. **Promoter Model** (`promoter_model`)
| Property | Details |
|----------|---------|
| **Task** | Binary promoter prediction |
| **Labels** | `Not promoter` / `Core promoter` |
| **Architecture** | DNABERT |
| **Tokenizer** | BPE |
| **Species** | Plant |
| **Model Source** | `zhangtaolab/plant-dnabert-BPE-promoter` (ModelScope) |
| **Performance** | Accuracy: **85%** · F1: **0.82** · Precision: **0.80** · Recall: **0.85** |
| **Memory Usage** | ~369.3 MB total (~351.7 MB parameters + ~17.6 MB activations) |
| **Status** | ✅ Loaded |

> Predicts whether a DNA sequence is a **core promoter** in plant species.

---

### 2. **Conservation Model** (`conservation_model`)
| Property | Details |
|----------|---------|
| **Task** | Binary conservation prediction |
| **Labels** | `Not conserved` / `Conserved` |
| **Architecture** | DNABERT |
| **Tokenizer** | BPE |
| **Sp